In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable
from pyspark.sql.window import Window

In [0]:
df = spark.table("pcat.silver.gross_price")
df = df.select("product_code", "month", "gross_price")

In [0]:
df.write \
 .format("delta") \
 .option("delta.enableChangeDataFeed", "true") \
 .mode("overwrite") \
 .saveAsTable("pcat.gold.sb_dim_gross_price")

In [0]:
df = spark.table("pcat.gold.sb_dim_gross_price")

In [0]:
df = (
    df
    .withColumn("year", F.year("month"))
    .withColumn("is_zero", F.when(F.col("gross_price") == 0, 1).otherwise(0))
)

window = (
    Window
    .partitionBy("product_code", "year")
    .orderBy(F.col("is_zero"), F.col("month").desc())
)

df_latest_price = (
    df
    .withColumn("rank", F.row_number().over(window))
    .filter(F.col("rank") == 1)
)

In [0]:
display(df_latest_price)

In [0]:
df_latest_price = df_latest_price.select("product_code", "year", "gross_price").withColumnRenamed("gross_price", "price_inr").select("product_code", "price_inr", "year")

df_latest_price = df_latest_price.withColumn("year", F.col("year").cast("string"))

In [0]:
df_latest_price.show(5)

In [0]:
delta_table = DeltaTable.forName(spark, "pcat.gold.dim_gross_price")
delta_table.alias("target").merge(
    source=df_latest_price.alias("source"),
    condition="target.product_code = source.product_code"
).whenMatchedUpdate(
    set={
        "price_inr": "source.price_inr",
        "year": "source.year"
    }
).whenNotMatchedInsert(
    values={
        "product_code": "source.product_code",
        "price_inr": "source.price_inr",
        "year": "source.year"
    }
).execute()